# Custom Tranches — Advanced LBO Modeling

This notebook demonstrates advanced tranche structures including 100% PIK, step-up coupons, and revolver modeling.

In [ ]:
import sys
sys.path.insert(0, '../src')

from lbo_simulator.models.debt_tranche import DebtTranche
from lbo_simulator.data.synthetic import SyntheticCompanyGenerator
from lbo_simulator.models.lbo_engine import LBOEngine
from lbo_simulator.models.schemas import (
    DebtTrancheSchema,
    SourcesAndUsesSchema,
    ExitAssumptionsSchema,
    LBOConfigSchema,
    CovenantThresholdsSchema,
    CompanyProfileSchema,
)
import pandas as pd

## 1. 100% PIK Structure

In a 100% PIK structure, no cash interest is paid — all interest accrues to principal.

In [ ]:
pik_tranche = DebtTranche(
    name="Pure PIK Mezzanine",
    tranche_type="mezzanine",
    principal=50_000_000,
    interest_rate=0.0,  # No cash interest
    amortization_rate=0.0,
    maturity_years=5.0,
    pik_toggle=True,
    pik_rate=0.12,  # 12% PIK
)

print("Year | Beg Balance | PIK Accrued | End Balance")
print("-" * 55)

for year in range(1, 6):
    result = pik_tranche.calculate_interest()
    print(
        f"{year:>4} | ${pik_tranche.outstanding_balance + result['pik_accrued']:>12,.0f} | "
        f"${result['pik_accrued']:>10,.0f} | ${result['ending_balance']:>10,.0f}"
    )
    pik_tranche._outstanding_balance = result['ending_balance']

## 2. Revolver with Drawdown/Repayment

In [ ]:
revolver = DebtTranche(
    name="Senior Revolver",
    tranche_type="revolver",
    principal=30_000_000,
    interest_rate=0.07,
    commitment_fee=0.005,  # 50bps on undrawn
    maturity_years=5.0,
)

# Calculate commitment fee on undrawn amount
drawn = 15_000_000
undrawn = revolver.principal - drawn
fee = revolver.calculate_commitment_fee(undrawn)

print(f"Revolver Commitment: ${revolver.principal:,.0f}")
print(f"Drawn: ${drawn:,.0f}")
print(f"Undrawn: ${undrawn:,.0f}")
print(f"Commitment Fee: ${fee:,.0f} ({revolver.commitment_fee:.2%} of undrawn)")

## 3. Cash Sweep Impact

Compare IRR with different sweep rates.

In [ ]:
def run_lbo_with_sweep(sweep_rate: float):
    gen = SyntheticCompanyGenerator(seed=42)
    company = gen.get_company_profile("SweepCo", sector="SaaS", initial_revenue=100_000_000)
    
    initial_ebitda = company.initial_revenue * company.initial_ebitda_margin
    purchase_price = initial_ebitda * 8.0
    
    return LBOConfigSchema(
        company=company,
        sources_and_uses=SourcesAndUsesSchema(
            equity_contribution=purchase_price * 0.40,
            senior_debt=purchase_price * 0.60,
            purchase_price=purchase_price,
        ),
        tranches=[
            DebtTrancheSchema(
                name="Senior TLB",
                tranche_type="senior_term_b",
                principal=purchase_price * 0.60,
                interest_rate=0.075,
                amortization_rate=0.01,
                maturity_years=7.0,
                cash_sweep_rate=sweep_rate,
            )
        ],
        exit_assumptions=ExitAssumptionsSchema(
            hold_period_years=5,
            exit_ebitda_multiple=9.0,
            entry_ebitda_multiple=8.0,
        ),
    )

sweep_rates = [0.0, 0.25, 0.50, 0.75, 1.0]
results = []

for sr in sweep_rates:
    config = run_lbo_with_sweep(sr)
    engine = LBOEngine(config)
    res = engine.run()
    results.append({
        'Sweep Rate': sr,
        'IRR': res.irr,
        'MOIC': res.moic,
        'Payback (y)': res.payback_period_years,
    })

pd.DataFrame(results).style.format({
    'Sweep Rate': '{:.0%}',
    'IRR': '{:.1%}',
    'MOIC': '{:.2f}x',
    'Payback (y)': '{:.1f}',
})

## 4. Multi-Tranche with Different Sweep Rates

In [ ]:
gen = SyntheticCompanyGenerator(seed=42)
company = gen.get_company_profile("MultiTrancheCo", sector="Industrials", initial_revenue=200_000_000)

initial_ebitda = company.initial_revenue * company.initial_ebitda_margin
purchase_price = initial_ebitda * 6.5
total_debt = purchase_price * 0.50

config = LBOConfigSchema(
    company=company,
    sources_and_uses=SourcesAndUsesSchema(
        equity_contribution=purchase_price * 0.50,
        senior_debt=total_debt * 0.70,
        mezzanine_debt=total_debt * 0.30,
        purchase_price=purchase_price,
    ),
    tranches=[
        DebtTrancheSchema(
            name="Senior TLB",
            tranche_type="senior_term_b",
            principal=total_debt * 0.70,
            interest_rate=0.075,
            amortization_rate=0.01,
            maturity_years=7.0,
            cash_sweep_rate=0.50,  # 50% sweep
        ),
        DebtTrancheSchema(
            name="Mezzanine",
            tranche_type="mezzanine",
            principal=total_debt * 0.30,
            interest_rate=0.10,
            amortization_rate=0.0,
            maturity_years=5.0,
            cash_sweep_rate=0.0,  # No sweep on mezz
            pik_toggle=True,
            pik_rate=0.02,
        ),
    ],
    exit_assumptions=ExitAssumptionsSchema(
        hold_period_years=5,
        exit_ebitda_multiple=7.5,
        entry_ebitda_multiple=6.5,
    ),
)

engine = LBOEngine(config)
results = engine.run()

print(f"IRR:  {results.irr:.1%}")
print(f"MOIC: {results.moic:.2f}x")

# Debt schedule by tranche
debt_df = pd.DataFrame([
    {
        'Year': ds.year,
        'Tranche': ds.tranche_name,
        'End Balance ($M)': ds.ending_balance / 1e6,
        'Interest ($M)': ds.interest_paid / 1e6,
        'Amort ($M)': ds.mandatory_amortization / 1e6,
    }
    for ds in results.debt_schedule
])
debt_df